### Import

In [1]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 20
LEVEL = "high"
SEED = 42

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)

✅ 총 5개 파일을 불러왔습니다: 1201.csv, 137.csv, 401.csv, 524.csv, 89.csv
📊 데이터 Shape: I=5, T=24, S=20
✅ 시뮬레이션 초기화 완료: S=20, Randomness='high', Random Seed=42, M1=763.86, M2=2075.61


### Linear Decision Rule (Individual Optimization)

In [ ]:
model = gp.Model("individual_LDR")
model.setParam("MIPGap", 1e-5)
model.setParam(GRB.Param.PoolSearchMode, 2)
model.setParam(GRB.Param.PoolSolutions, 1)

x_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_ind = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp")
ym_ind = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
z_ind = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc_ind = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc")
zd_ind = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    
phi1_ind = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2_ind = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") ; phi3_ind = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3")

# LDR 계수 변수들
zc0_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc0") ; zc1_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc1") ; zc2_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc2")
zd0_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd0") ; zd1_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd1") ; zd2_ind = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd2")

model.update()

# obj = (gp.quicksum(P_DA[t] * x_ind[i, t] for i in range(I) for t in range(T)) + 
#        gp.quicksum((1/S) * (P_RT[t, s] * yp_ind[i, t, s] - P_PN[t, s] * ym_ind[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
# model.setObjective(obj, GRB.MAXIMIZE)

# # Quadratic Regularization
obj = (gp.quicksum(P_DA[t] * x_ind[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_ind[i, t, s] - P_PN[t, s] * ym_ind[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
reg = gp.quicksum(x_ind[i, t] * x_ind[i, t] for i in range(I) for t in range(T))
epsilon = 1e-7
regularized_obj = obj - epsilon * reg
model.setObjective(regularized_obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    # model.addConstr(zc_ind[i, t, s] == zc0_ind[i, t] + zc1_ind[i, t] * R[i, t, s])
    # model.addConstr(zd_ind[i, t, s] == zd0_ind[i, t] + zd1_ind[i, t] * R[i, t, s])
    # model.addConstr(zc_ind[i, t, s] == zc0_ind[i, t] + zc1_ind[i, t] * P_RT[t, s])
    # model.addConstr(zd_ind[i, t, s] == zd0_ind[i, t] + zd1_ind[i, t] * P_RT[t, s])
    # model.addConstr(zc_ind[i, t, s] == zc0_ind[i, t] + zc1_ind[i, t] * P_PN[t, s])
    # model.addConstr(zd_ind[i, t, s] == zd0_ind[i, t] + zd1_ind[i, t] * P_PN[t, s])
    model.addConstr(zc_ind[i, t, s] == zc0_ind[i, t] + zc1_ind[i, t] * (P_RT[t, s] - P_DA[t]))
    model.addConstr(zd_ind[i, t, s] == zd0_ind[i, t] + zd1_ind[i, t] * (P_RT[t, s] - P_DA[t]))
    # model.addConstr(zc_ind[i, t, s] == zc0_ind[i, t] + zc1_ind[i, t] * (P_PN[t, s] - P_DA[t]))
    # model.addConstr(zd_ind[i, t, s] == zd0_ind[i, t] + zd1_ind[i, t] * (P_PN[t, s] - P_DA[t]))
    # model.addConstr(zc_ind[i, t, s] == zc0_ind[i, t] + zc1_ind[i, t] * R[i, t, s] + zc2_ind[i, t] * P_RT[t, s])
    # model.addConstr(zd_ind[i, t, s] == zd0_ind[i, t] + zd1_ind[i, t] * R[i, t, s] + zd2_ind[i, t] * P_RT[t, s])
    # model.addConstr(zc_ind[i, t, s] == zc0_ind[i, t] + zc1_ind[i, t] * R[i, t, s] + zc2_ind[i, t] * P_PN[t, s])
    # model.addConstr(zd_ind[i, t, s] == zd0_ind[i, t] + zd1_ind[i, t] * R[i, t, s] + zd2_ind[i, t] * P_PN[t, s])
    # model.addConstr(zc_ind[i, t, s] == zc0_ind[i, t] + zc1_ind[i, t] * R[i, t, s] + zc2_ind[i, t] * (P_RT[t, s] - P_DA[t]))
    # model.addConstr(zd_ind[i, t, s] == zd0_ind[i, t] + zd1_ind[i, t] * R[i, t, s] + zd2_ind[i, t] * (P_RT[t, s] - P_DA[t]))


for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(R[i, t, s] - x_ind[i, t] == yp_ind[i, t, s] - ym_ind[i, t, s] + zc_ind[i, t, s] - zd_ind[i, t, s])
    model.addConstr(R[i, t, s] + zd_ind[i, t, s] >= yp_ind[i, t, s] + zc_ind[i, t, s])
    model.addConstr(zd_ind[i, t, s] <= z_ind[i, t, s])
    model.addConstr(zc_ind[i, t, s] <= K[i] - z_ind[i, t, s])
    model.addConstr(z_ind[i, t, s] <= K[i])
    model.addConstr(z_ind[i, t + 1, s] == z_ind[i, t, s] + 0.9 * zc_ind[i, t, s] - zd_ind[i, t, s] / 0.9)
        
    model.addConstr(yp_ind[i, t, s] <= M1 * phi1_ind[i, t, s]) ; model.addConstr(ym_ind[i, t, s] <= M1 * (1 - phi1_ind[i, t, s]))
    model.addConstr(ym_ind[i, t, s] <= M1 * phi2_ind[i, t, s]) ; model.addConstr(zc_ind[i, t, s] <= M1 * (1 - phi2_ind[i, t, s]))
    model.addConstr(zc_ind[i, t, s] <= M1 * phi3_ind[i, t, s]) ; model.addConstr(zd_ind[i, t, s] <= M1 * (1 - phi3_ind[i, t, s]))
    
for i, s in product(range(I), range(S)): model.addConstr(z_ind[i, 0, s] == K0[i])

model.optimize()

Set parameter MIPGap to value 1e-05
Set parameter PoolSearchMode to value 2


Set parameter PoolSolutions to value 2
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-05
PoolSolutions  2
PoolSearchMode  2

Optimize a model with 33700 rows, 20140 columns and 84100 nonzeros
Model fingerprint: 0xa1df85fd
Model has 120 quadratic objective terms
Variable types: 12940 continuous, 7200 integer (7200 binary)
Coefficient statistics:
  Matrix range     [5e-03, 8e+02]
  Objective range  [2e+00, 2e+02]
  QObjective range [2e-07, 2e-07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e-02, 8e+02]
Found heuristic solution: objective 868401.91111
Presolve removed 12153 rows and 4299 columns
Presolve time: 0.22s
Presolved: 21547 rows, 15841 columns, 57392 nonzeros
Presolved model has 120 quadratic objective terms
Variable types: 8641 continuous, 

In [ ]:
if model.status == GRB.OPTIMAL:
    num_solutions = model.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = model.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            model.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = model.PoolObjVal
            diff = best_obj - pool_obj
            
            print(f"Solution {i}: Objective = {pool_obj},  Difference from best = {diff}")

    model.setParam(GRB.Param.SolutionNumber, 0)
    
    x_ind = np.array([[x_ind[i, t].X for t in range(T)] for i in range(I)])
    yp_ind = np.array([[[yp_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_ind = np.array([[[ym_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_ind = np.array([[[zc_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_ind = np.array([[[zd_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_ind = np.array([[[z_ind[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = model.objVal
    phi1_ind = np.array([[[phi1_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_ind = np.array([[[phi2_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    phi3_ind = np.array([[[phi3_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    
    zc0_ind = np.array([[zc0_ind[i, t].X for t in range(T)] for i in range(I)]) ; zc1_ind = np.array([[zc1_ind[i, t].X for t in range(T)] for i in range(I)]) ; zc2_ind = np.array([[zc2_ind[i, t].X for t in range(T)] for i in range(I)])
    zd0_ind = np.array([[zd0_ind[i, t].X for t in range(T)] for i in range(I)]) ; zd1_ind = np.array([[zd1_ind[i, t].X for t in range(T)] for i in range(I)]) ; zd2_ind = np.array([[zd2_ind[i, t].X for t in range(T)] for i in range(I)])
    OBJ_IND = model.objVal


--- Solution Pool Analysis ---
Found 2 solutions in the pool.
Best objective value: 1632097.93227715

Solution 0: Objective = 1632097.9322771467,  Difference from best = 0.0
Solution 1: Objective = 1632097.932249691,  Difference from best = 2.745562233030796e-05


In [21]:
for i, t, s in product(range(I), range(T), range(S)):
    if yp_ind[i, t, s] != 0 and ym_ind[i, t, s] != 0: print("[VIOLATION] YP-YM", i, t, s, yp_ind[i,t,s], ym_ind[i,t,s])
    if ym_ind[i, t, s] != 0 and zc_ind[i, t, s] != 0: print("[VIOLATION] YM-ZC", i, t, s, ym_ind[i,t,s], zc_ind[i,t,s])
    if zc_ind[i, t, s] != 0 and zd_ind[i, t, s] != 0: print("[VIOLATION] ZC-ZD", i, t, s, zc_ind[i,t,s], zd_ind[i,t,s])

In [22]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 70)
print("\n[Individual]") ; print(header)
for t in range(7, 22):
    # R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_ind[:, t].sum()
    # yp_avg = np.mean([yp_ind[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_ind[:, t, s].sum() for s in range(S)])
    # zc_avg = np.mean([zc_ind[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_ind[:, t, s].sum() for s in range(S)]) 
    # z_avg = np.mean([z_ind[:, t, s].sum() for s in range(S)])

    i=0
    R_avg = np.mean([R[i, t, s] for s in range(S)]) ; x_sum = x_ind[i, t]
    yp_avg = np.mean([yp_ind[i, t, s] for s in range(S)]) ; ym_avg = np.mean([ym_ind[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_ind[i, t, s] for s in range(S)]) ; zd_avg = np.mean([zd_ind[i, t, s] for s in range(S)]) 
    z_avg = np.mean([z_ind[i, t, s] for s in range(S)])

    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[Individual]
 t |        R        x       y+       y-       zc       zd        z
----------------------------------------------------------------------
 7 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 8 |    15.70     0.00     3.15     0.00    12.55     0.00     0.00
 9 |     3.87     0.00     0.00     0.00     3.87     0.00    11.30
10 |    33.49     0.00    18.20     0.00    15.29     0.00    14.78
11 |    60.71     0.00    35.83     0.00    24.88     0.00    28.54
12 |   166.49     0.00   134.69     0.00    31.80     0.00    50.93
13 |   256.48     0.00   304.77     0.00     0.00    48.29    79.55
14 |   163.84     0.00   109.89     0.00    53.95     0.00    25.89
15 |   160.46     0.00   220.73     0.00     0.00    60.27    74.45
16 |    18.71     7.13    11.88     0.31     0.00     0.00     7.48
17 |     0.00     0.00     0.00     0.00     0.00     0.00     7.48
18 |    67.36    17.16    50.20     0.00     0.00     0.00     7.48
19 |    67.32    36.55    31.97

In [ ]:
print("\n=== 모든 LDR 계수 (i, t별) ===")
print("Individual | Time | Variable | 상수항    | R계수    ")
print("-" * 50)

for t in range(9,20):
    for i in range(I):
        print(f"{i:10d} | {t:4d} | zc       | {zc0_ind[i,t]:8.4f} | {zc1_ind[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | zd       | {zd0_ind[i,t]:8.4f} | {zd1_ind[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | x (1st)  | {x_ind[i,t]:8.4f} |    -    ")
        print("-" * 50)


=== 모든 LDR 계수 (i, t별) ===
Individual | Time | Variable | 상수항    | R계수    
--------------------------------------------------
         0 |    9 | zc       |   5.8829 |  0.0774
         0 |    9 | zd       |   0.0000 |  0.0000
         0 |    9 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         1 |    9 | zc       |  18.0606 |  0.0000
         1 |    9 | zd       |   0.0000 |  0.0000
         1 |    9 | x (1st)  |   0.0001 |    -    
--------------------------------------------------
         2 |    9 | zc       |  18.6331 |  0.0000
         2 |    9 | zd       |   0.0000 |  0.0000
         2 |    9 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         3 |    9 | zc       |   5.5543 |  0.0000
         3 |    9 | zd       |   0.0000 |  0.0000
         3 |    9 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         4 |    9 | zc       |   6.0872 |  0.0000
         4 |    

### Linear Decision Rule (Holistic Optimization)

In [7]:
model = gp.Model("holistic_LDR")
model.setParam("MIPGap", 1e-5)
model.setParam(GRB.Param.PoolSearchMode, 2)
model.setParam(GRB.Param.PoolSolutions, 2)

x_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z_hol = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    
phi1_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
phi3_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
phi5_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
phi7_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")

zc0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc0") ; zcR_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zcR")
zd0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd0") ; zdR_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zdR")
zc2_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd2") ; zd2_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd2")

model.update()

# obj = (gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + 
#        gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
# model.setObjective(obj, GRB.MAXIMIZE)

# # Quadratic Regularization
obj = (gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
reg = gp.quicksum(x_hol[i, t] * x_hol[i, t] for i in range(I) for t in range(T))
epsilon = 1e-7
regularized_obj = obj - epsilon * reg
model.setObjective(regularized_obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    # model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zcR_hol[i, t] * R[i, t, s])
    # model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zdR_hol[i, t] * R[i, t, s])
    # model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zcR_hol[i, t] * P_RT[t, s])
    # model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zdR_hol[i, t] * P_RT[t, s])
    # model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zcR_hol[i, t] * P_PN[t, s])
    # model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zdR_hol[i, t] * P_PN[t, s])
    model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zcR_hol[i, t] * (P_RT[t, s] - P_DA[t]))
    model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zdR_hol[i, t] * (P_RT[t, s] - P_DA[t]))
    # model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zcR_hol[i, t] * (P_PN[t, s] - P_DA[t]))
    # model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zdR_hol[i, t] * (P_PN[t, s] - P_DA[t]))
    # model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zcR_hol[i, t] * R[i, t, s] + zc2_hol[i, t] * P_RT[t, s])
    # model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zdR_hol[i, t] * R[i, t, s] + zd2_hol[i, t] * P_RT[t, s])
    # model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zcR_hol[i, t] * R[i, t, s] + zc2_hol[i, t] * P_PN[t, s])
    # model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zdR_hol[i, t] * R[i, t, s] + zd2_hol[i, t] * P_PN[t, s])
    # model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zcR_hol[i, t] * R[i, t, s] + zc2_hol[i, t] * (P_RT[t, s] - P_DA[t]))
    # model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zdR_hol[i, t] * R[i, t, s] + zd2_hol[i, t] * (P_RT[t, s] - P_DA[t]))

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
    model.addConstr(R[i, t, s] + zd_hol[i, t, s] >= yp_hol[i, t, s] + dp_hol[i, t, s] + zc_hol[i, t, s])
    model.addConstr(zd_hol[i, t, s] <= z_hol[i, t, s])
    model.addConstr(zc_hol[i, t, s] <= K[i] - z_hol[i, t, s])
    model.addConstr(z_hol[i, t, s] <= K[i])
    model.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + 0.9 * zc_hol[i, t, s] - zd_hol[i, t, s] / 0.9)
        
    model.addConstr(yp_hol[i, t, s] <= M1 * phi1_hol[i, t, s]) ; model.addConstr(ym_hol[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
    model.addConstr(dp_hol[i, t, s] <= M1 * phi2_hol[i, t, s]) ; model.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
    model.addConstr(yp_hol[i, t, s] <= M1 * phi3_hol[i, t, s]) ; model.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
    model.addConstr(ym_hol[i, t, s] <= M1 * phi4_hol[i, t, s]) ; model.addConstr(dp_hol[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
    model.addConstr(ym_hol[i, t, s] <= M1 * phi5_hol[i, t, s]) ; model.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
    model.addConstr(dm_hol[i, t, s] <= M1 * phi6_hol[i, t, s]) ; model.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))
    model.addConstr(zc_hol[i, t, s] <= M1 * phi7_hol[i, t, s]) ; model.addConstr(zd_hol[i, t, s] <= M1 * (1 - phi7_hol[i, t, s]))
    
for i, s in product(range(I), range(S)): model.addConstr(z_hol[i, 0, s] == K0[i])

balance_constraints = {}
for t, s in product(range(T), range(S)):
    balance_constraints[t, s] = model.addConstr(gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)), name=f"balance_{t}_{s}")

model.optimize()

Set parameter MIPGap to value 1e-05
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 2
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-05
PoolSolutions  2
PoolSearchMode  2

Optimize a model with 53380 rows, 34540 columns and 134500 nonzeros
Model fingerprint: 0x5c550ae8
Model has 120 quadratic objective terms
Variable types: 17740 continuous, 16800 integer (16800 binary)
Coefficient statistics:
  Matrix range     [5e-03, 8e+02]
  Objective range  [2e+00, 2e+02]
  QObjective range [2e-07, 2e-07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e-02, 8e+02]
Presolve removed 16146 rows and 5132 columns
Presolve time: 0.26s
Presolved: 37234 rows, 29408 columns, 95840 nonzeros
Presolved model has 120 quadratic objective terms
Var

In [8]:
if model.status == GRB.OPTIMAL:
    num_solutions = model.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = model.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            model.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = model.PoolObjVal
            diff = best_obj - pool_obj
            
            print(f"Solution {i}: Objective = {pool_obj},  Difference from best = {diff}")

    model.setParam(GRB.Param.SolutionNumber, 0)
    
    x_hol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
    yp_hol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_hol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_hol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_hol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_hol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_hol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_hol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = model.objVal
    phi1_hol = np.array([[[phi1_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_hol = np.array([[[phi2_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    phi3_hol = np.array([[[phi3_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi4_hol = np.array([[[phi4_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    phi5_hol = np.array([[[phi5_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi6_hol = np.array([[[phi6_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    phi7_hol = np.array([[[phi7_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; OBJ_HOL = model.objVal
    
    zc0_hol = np.array([[zc0_hol[i, t].X for t in range(T)] for i in range(I)]) ; zcR_hol = np.array([[zcR_hol[i, t].X for t in range(T)] for i in range(I)])
    zd0_hol = np.array([[zd0_hol[i, t].X for t in range(T)] for i in range(I)]) ; zdR_hol = np.array([[zdR_hol[i, t].X for t in range(T)] for i in range(I)])


--- Solution Pool Analysis ---
Found 2 solutions in the pool.
Best objective value: 1689908.85155376

Solution 0: Objective = 1689908.8515537635,  Difference from best = 0.0
Solution 1: Objective = 1689908.8515645755,  Difference from best = -1.0811956599354744e-05


In [9]:
if model.status == GRB.OPTIMAL:        
    try:
        lambda_dual = {}
        for t, s in product(range(T), range(S)): 
            lambda_dual[t, s] = balance_constraints[t, s].Pi
        print("Direct dual extraction successful!")
        
    except AttributeError:
        print("\nDirect dual extraction failed. Using Model.fixed() method...")
        
        fixed_model = model.fixed()
        
        fixed_x_hol = {(i, t): fixed_model.getVarByName(f"x[{i},{t}]") for i, t in product(range(I), range(T))}
        fixed_yp_hol = {(i, t, s): fixed_model.getVarByName(f"yp[{i},{t},{s}]") for i, t, s in product(range(I), range(T), range(S))}
        fixed_ym_hol = {(i, t, s): fixed_model.getVarByName(f"ym[{i},{t},{s}]") for i, t, s in product(range(I), range(T), range(S))}

        linear_obj_for_fixed_model = (
            gp.quicksum(P_DA[t] * fixed_x_hol[i, t] for i, t in product(range(I), range(T))) +
            gp.quicksum((1/S) * (P_RT[t, s] * fixed_yp_hol[i, t, s] - P_PN[t, s] * fixed_ym_hol[i, t, s]) 
                         for i, t, s in product(range(I), range(T), range(S)))
        )
        
        reg_fixed = gp.quicksum(fixed_x_hol[i, t] * fixed_x_hol[i, t] for i, t in product(range(I), range(T)))
        regularized_obj_fixed = linear_obj_for_fixed_model - epsilon * reg_fixed
        # regularized_obj_fixed = linear_obj_for_fixed_model
        
        fixed_model.setParam("Method", 1)
        fixed_model.setParam("MIPGap", 1e-5)
        fixed_model.setParam(GRB.Param.PoolSearchMode, 2)
        fixed_model.setParam(GRB.Param.PoolSolutions, 2)
        fixed_model.setObjective(regularized_obj_fixed, GRB.MAXIMIZE)
        fixed_model.optimize()
        
        if fixed_model.status == GRB.OPTIMAL:
            original_obj_val = (
                sum(P_DA[t] * x_hol[i, t] for i, t in product(range(I), range(T))) +
                sum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) 
                    for i, t, s in product(range(I), range(T), range(S)))
                - epsilon * sum(x_hol[i, t] * x_hol[i, t] for i, t in product(range(I), range(T)))
            )
            
            fixed_obj = fixed_model.objVal
            obj_diff = abs(original_obj_val - fixed_obj)

            print(f"Original Objective: {original_obj_val:.6f}")
            print(f"Fixed Model Objective: {fixed_obj:.6f}")
            print(f"Difference: {obj_diff:.10f}")
            
            if obj_diff < 1e-6:
                print("✅ Objective values match! Fixed model is consistent.")
            else:
                print("⚠️ Warning: Objective values don't match.")

            num_solutions = fixed_model.SolCount
            print(f"\n--- Fixed Model Solution Pool Analysis ---")
            print(f"Found {num_solutions} solutions in the pool.")

            if num_solutions > 1:
                best_obj = fixed_model.objVal
                print(f"Best objective value: {best_obj:.8f}\n")
                for i in range(num_solutions):
                    fixed_model.setParam(GRB.Param.SolutionNumber, i)
                    pool_obj = fixed_model.PoolObjVal
                    diff = abs(best_obj - pool_obj)
                    print(f"Solution {i}: Objective = {pool_obj:.8f},  Difference = {diff:.8f}")
                fixed_model.setParam(GRB.Param.SolutionNumber, 0)

            print("\n=== Solution Comparison ===")
            fixed_vars = {var.VarName: var.X for var in fixed_model.getVars()}
            
            print("=== x values by individual ===")
            print("Individual | Time | Original x | Fixed x | Difference")
            print("-" * 55)
            
            max_x_diff = 0
            for i in range(I):
                for t in range(T):
                    original_x = x_hol[i, t]
                    fixed_x = fixed_vars.get(f"x[{i},{t}]", 0)
                    diff = abs(original_x - fixed_x)
                    max_x_diff = max(max_x_diff, diff)
                    if original_x != 0:
                        print(f"{i:10d} | {t:4d} | {original_x:10.6f} | {fixed_x:7.6f} | {diff:10.8f}")
            
            print("\n=== yp values sum over i (scenario average) ===")
            print("Time | Original yp_sum | Fixed yp_sum | Difference")
            print("-" * 52)
            max_yp_diff = 0
            for t in range(14,16):
                original_yp_sum = sum(sum(yp_hol[i, t, s] for i in range(I)) for s in range(S)) / S
                fixed_yp_sum = sum(sum(fixed_vars.get(f"yp[{i},{t},{s}]", 0) for i in range(I)) for s in range(S)) / S
                diff = abs(original_yp_sum - fixed_yp_sum)
                max_yp_diff = max(max_yp_diff, diff)
                print(f"{t:4d} | {original_yp_sum:14.6f} | {fixed_yp_sum:12.6f} | {diff:10.8f}")

        lambda_dual = np.zeros((T, S))
        if fixed_model.status == GRB.OPTIMAL:
            for t, s in product(range(T), range(S)):
                constr_name = f"balance_{t}_{s}"
                try:
                    constr = fixed_model.getConstrByName(constr_name)
                    if constr is not None:
                        lambda_dual[t, s] = constr.Pi
                    else:
                        lambda_dual[t, s] = np.nan
                except:
                    lambda_dual[t, s] = np.nan 
                    
            print("\nModel.fixed() dual extraction successful!")
        else:
            print("Fixed model optimization failed. Setting dual to zeros.")
            lambda_dual = np.zeros((T, S))

        print("\nInternal Settlement Prices (Dual Variables):")
        for t, s in product(range(T), range(1,2)):
            print(f"λ_{t}(ξ_{s}) = {lambda_dual[t, s]:.4f}")
            
else:
    print("No optimal solution found.")
    lambda_dual = {(t, s): np.nan for t in range(T) for s in range(S)}
    x_hol = yp_hol = ym_hol = dp_hol = dm_hol = z_hol = zc_hol = zd_hol = None
    original_objval = None


Direct dual extraction failed. Using Model.fixed() method...
Set parameter Method to value 1
Set parameter MIPGap to value 1e-05
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 2
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-05
Method  1
PoolSolutions  2
PoolSearchMode  2

Optimize a model with 53380 rows, 34540 columns and 134500 nonzeros
Model fingerprint: 0x6f4a0359
Model has 120 quadratic objective terms
Coefficient statistics:
  Matrix range     [5e-03, 8e+02]
  Objective range  [2e+00, 2e+02]
  QObjective range [2e-07, 2e-07]
  Bounds range     [1e-06, 1e+00]
  RHS range        [8e-02, 8e+02]
Presolve time: 0.03s
Presolved: 3138 rows, 2368 columns, 11394 nonzeros
Presolved model has 91 quadratic objective terms

Iteration

In [10]:
for i, t, s in product(range(I), range(T), range(S)):
    if yp_hol[i, t, s] != 0 and ym_hol[i, t, s] != 0: print("[VIOLATION] YP-YM", i, t, s, yp_hol[i,t,s], ym_hol[i,t,s])
    if dp_hol[i, t, s] != 0 and dm_hol[i, t, s] != 0: print("[VIOLATION] DP-DM", i, t, s, dp_hol[i,t,s], dm_hol[i,t,s])
    if dp_hol[i, t, s] != 0 and ym_hol[i, t, s] != 0: print("[VIOLATION] DP-YM", i, t, s, dp_hol[i,t,s], ym_hol[i,t,s])
    if yp_hol[i, t, s] != 0 and dm_hol[i, t, s] != 0: print("[VIOLATION] YP-DM", i, t, s, yp_hol[i,t,s], dm_hol[i,t,s])
    if ym_hol[i, t, s] != 0 and zc_hol[i, t, s] != 0: print("[VIOLATION] YM-ZC", i, t, s, ym_hol[i,t,s], zc_hol[i,t,s])
    if dm_hol[i, t, s] != 0 and zc_hol[i, t, s] != 0: print("[VIOLATION] DM-ZC", i, t, s, dm_hol[i,t,s], zc_hol[i,t,s])
    if zc_hol[i, t, s] != 0 and zd_hol[i, t, s] != 0: print("[VIOLATION] ZC-ZD", i, t, s, zc_hol[i,t,s], zd_hol[i,t,s])

In [11]:
print("\n=== 모든 LDR 계수 (i, t별) ===")
print("Individual | Time | Variable | 상수항     | R계수    ")
print("-" * 50)

for t in range(7,20):
    for i in range(I):
        print(f"{i:10d} | {t:4d} | zc       | {zc0_hol[i,t]:8.4f} | {zcR_hol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | zd       | {zd0_hol[i,t]:8.4f} | {zdR_hol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | x (1st)  | {x_hol[i,t]:8.4f} |    -    ")
        print("-" * 50)


=== 모든 LDR 계수 (i, t별) ===
Individual | Time | Variable | 상수항     | R계수    
--------------------------------------------------
         0 |    7 | zc       |   0.0000 |  0.0000
         0 |    7 | zd       |   0.0000 |  0.0000
         0 |    7 | x (1st)  |   4.8893 |    -    
--------------------------------------------------
         1 |    7 | zc       |   1.1641 |  0.0000
         1 |    7 | zd       |   0.0000 |  0.0000
         1 |    7 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         2 |    7 | zc       |   1.4627 |  0.0000
         2 |    7 | zd       |   0.0000 |  0.0000
         2 |    7 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         3 |    7 | zc       |   0.0818 |  0.0000
         3 |    7 | zd       |   0.0000 |  0.0000
         3 |    7 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         4 |    7 | zc       |   0.9202 |  0.0000
         4 |   

In [12]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print("\n[HOLISTIC]") ; print(header)
for t in range(7, 22):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[HOLISTIC]
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 7 |    12.59     4.89     4.30     0.23     4.66     4.66     3.63     0.00     1.00
 8 |    41.70    18.42    12.62     0.51    16.10    16.10    11.17     0.00     4.26
 9 |   185.31   113.96    29.72     6.70   103.39   103.39    48.34     0.00    14.31
10 |   445.25   220.98   117.68     2.04   185.45   185.45   108.62     0.00    57.82
11 |   652.06   255.20   241.14     0.00   183.50   183.50   155.71     0.00   155.58
12 |   860.69     0.00   752.36     0.00     0.00     0.00   108.33     0.00   295.72
13 |  1397.66     0.00  1556.27     0.00     0.00     0.00     0.00   158.61   393.22
14 |  1338.06   914.59   179.83    20.67   508.43   508.43   264.31     0.00   216.98
15 |   804.05     0.00  1213.42     0.00     0.00     0.00     0.00   409.38   454.87
16 |   756.56   538.86   223.64     5

In [13]:
data = []
for t, s in product(range(T), range(1,4)):
    data.append({
        'Time': t,
        'Scenario': s,
        'P_DA': round(P_DA[t], 2),
        'P_RT': round(P_RT[t, s], 2),
        'Dual': round(-lambda_dual[t, s] * S, 2),
        'P_PN': round(P_PN[t, s], 2)
    })
pd.DataFrame(data)

,Time,Scenario,P_DA,P_RT,Dual,P_PN
0,0,1,106.720,84.830,-0.000,213.440
1,0,2,106.720,74.950,-0.000,213.440
2,0,3,106.720,68.930,-0.000,213.440
3,1,1,93.130,67.580,-0.000,186.260
4,1,2,93.130,77.240,-0.000,186.260
5,1,3,93.130,81.930,-0.000,186.260
6,2,1,86.030,75.110,-0.000,172.060
7,2,2,86.030,50.800,-0.000,172.060
8,2,3,86.030,96.960,-0.000,193.920
9,3,1,82.960,57.710,-0.000,165.930


In [33]:
lambda_rep = np.zeros((T, S))

data = []
for t in range(T):
    for s in range(S):
        lambda_rep[t, s] = lambda_dual[t, s]
    p_rt_avg = np.mean(P_RT[t, :]) ; p_pn_avg = np.mean(P_PN[t, :])
    
    # lambda_rep[t, :] = np.mean(lambda_rep[t, :])
    lambda_rep[t, :] = np.mean(lambda_rep[t, :][lambda_rep[t, :] < 0]) if np.any(lambda_rep[t, :] < 0) else 0
    # lambda_rep[t, :] = np.mean(lambda_rep[t, :][np.abs(lambda_rep[t, :]) > 0]) if np.any(np.abs(lambda_rep[t, :]) > 0) else 0
    
    data.append({
        'Time': t, 'P_RT_avg': round(p_rt_avg, 2), 'Lambda': round(-lambda_rep[t, 0] * S, 2), 'P_PN_avg': round(p_pn_avg, 2)
    })
pd.DataFrame(data)

,Time,P_RT_avg,Lambda,P_PN_avg
0,0,62.570,2134.440,213.440
1,1,87.530,1862.560,196.750
2,2,74.840,1720.600,178.380
3,3,66.830,1659.280,167.120
4,4,72.650,78.370,168.650
5,5,86.840,1718.920,188.890
6,6,71.170,93.550,187.240
7,7,86.800,102.330,209.710
8,8,100.300,122.120,246.570
9,9,106.080,132.150,264.290


### Individual Replay

In [43]:
model = gp.Model("DER_Individual_Replay")
model.setParam("MIPGap", 1e-5)
model.setParam(GRB.Param.PoolSearchMode, 2)
model.setParam(GRB.Param.PoolSolutions, 2)

x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym") 
dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm") 
z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
phi1 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
phi3 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
phi5 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6") ; phi7 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")

zc0 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc0") ; zcR = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zcR")
zd0 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd0") ; zdR = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zdR")
zc2 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc2") ; zd2 = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd2")

model.update()

obj = (
    gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) + 
    gp.quicksum((1/S) * (
        P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
    ) for i in range(I) for t in range(T) for s in range(S)) +
    gp.quicksum(lambda_rep[t, s] * (
        gp.quicksum(dm[i, t, s] for i in range(I)) - gp.quicksum(dp[i, t, s] for i in range(I))
    ) for t in range(T) for s in range(S))
)
# model.setObjective(obj, GRB.MAXIMIZE)

epsilon = 1e-7
reg = gp.quicksum(x[i, t] * x[i, t] for i in range(I) for t in range(T))
regularized_obj = obj - epsilon * reg
model.setObjective(regularized_obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    # model.addConstr(zc[i, t, s] == zc0[i, t] + zcR[i, t] * R[i, t, s])
    # model.addConstr(zd[i, t, s] == zd0[i, t] + zdR[i, t] * R[i, t, s])
    # model.addConstr(zc[i, t, s] == zc0[i, t] + zcR[i, t] * P_RT[t, s])
    # model.addConstr(zd[i, t, s] == zd0[i, t] + zdR[i, t] * P_RT[t, s])
    # model.addConstr(zc[i, t, s] == zc0[i, t] + zcR[i, t] * P_PN[t, s])
    # model.addConstr(zd[i, t, s] == zd0[i, t] + zdR[i, t] * P_PN[t, s])
    model.addConstr(zc[i, t, s] == zc0[i, t] + zcR[i, t] * (P_RT[t, s] - P_DA[t]))
    model.addConstr(zd[i, t, s] == zd0[i, t] + zdR[i, t] * (P_RT[t, s] - P_DA[t]))
    # model.addConstr(zc[i, t, s] == zc0[i, t] + zcR[i, t] * (P_PN[t, s] - P_DA[t]))
    # model.addConstr(zd[i, t, s] == zd0[i, t] + zdR[i, t] * (P_PN[t, s] - P_DA[t]))
    # model.addConstr(zc[i, t, s] == zc0[i, t] + zcR[i, t] * R[i, t, s] + zc2[i, t] * P_RT[t, s])
    # model.addConstr(zd[i, t, s] == zd0[i, t] + zdR[i, t] * R[i, t, s] + zd2[i, t] * P_RT[t, s])
    # model.addConstr(zc[i, t, s] == zc0[i, t] + zcR[i, t] * R[i, t, s] + zc2[i, t] * P_PN[t, s])
    # model.addConstr(zd[i, t, s] == zd0[i, t] + zdR[i, t] * R[i, t, s] + zd2[i, t] * P_PN[t, s])
    # model.addConstr(zc[i, t, s] == zc0[i, t] + zcR[i, t] * R[i, t, s] + zc2[i, t] * (P_RT[t, s] - P_DA[t]))
    # model.addConstr(zd[i, t, s] == zd0[i, t] + zdR[i, t] * R[i, t, s] + zd2[i, t] * (P_RT[t, s] - P_DA[t]))

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
    model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
    model.addConstr(zd[i, t, s] <= z[i, t, s]) ; model.addConstr(zc[i, t, s] <= K[i] - z[i, t, s]) ; model.addConstr(z[i, t, s] <= K[i])
    model.addConstr(z[i, t + 1, s] == z[i, t, s] + 0.9 * zc[i, t, s] - zd[i, t, s] / 0.9)
    
    model.addConstr(yp[i, t, s] <= M1 * phi1_hol[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
    model.addConstr(dp[i, t, s] <= M1 * phi2_hol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
    model.addConstr(yp[i, t, s] <= M1 * phi3_hol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
    model.addConstr(ym[i, t, s] <= M1 * phi4_hol[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
    model.addConstr(ym[i, t, s] <= M1 * phi5_hol[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
    model.addConstr(dm[i, t, s] <= M1 * phi6_hol[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))
    model.addConstr(zc[i, t, s] <= M1 * phi7_hol[i, t, s]) ; model.addConstr(zd[i, t, s] <= M1 * (1 - phi7_hol[i, t, s]))
    
    # model.addConstr(yp[i, t, s] <= M1 * phi1[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1[i, t, s]))
    # model.addConstr(dp[i, t, s] <= M1 * phi2[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi2[i, t, s]))
    # model.addConstr(yp[i, t, s] <= M1 * phi3[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi3[i, t, s]))
    # model.addConstr(ym[i, t, s] <= M1 * phi4[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi4[i, t, s]))
    # model.addConstr(ym[i, t, s] <= M1 * phi5[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi5[i, t, s]))
    # model.addConstr(dm[i, t, s] <= M1 * phi6[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi6[i, t, s]))
    # model.addConstr(zc[i, t, s] <= M1 * phi7[i, t, s]) ; model.addConstr(zd[i, t, s] <= M1 * (1 - phi7[i, t, s]))

for i, s in product(range(I), range(S)): 
    model.addConstr(z[i, 0, s] == K0[i])
    # model.addConstr(dp[i,12,s] == 0) ; model.addConstr(dp[i,13,s] == 0) ; model.addConstr(dp[i,15,s] == 0)
    # model.addConstr(dm[i,12,s] == 0) ; model.addConstr(dm[i,13,s] == 0) ; model.addConstr(dm[i,15,s] == 0)

model.optimize()

if model.status == GRB.OPTIMAL:
    num_solutions = model.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = model.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            model.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = model.PoolObjVal
            diff = best_obj - pool_obj
            
            print(f"Solution {i}: Objective = {pool_obj},  Difference from best = {diff}")

    model.setParam(GRB.Param.SolutionNumber, 0)
    
    print(f"Optimal solution found! Objective value: {model.objVal}")
else:
    print("No optimal solution found.")

x_re = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_re = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_re = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
dp_re = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_re = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_re = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
zc_re = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_re = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
zc0_re = np.array([[zc0[i, t].X for t in range(T)] for i in range(I)]) ; zd0_re = np.array([[zd0[i, t].X for t in range(T)] for i in range(I)])
zcR_re = np.array([[zcR[i, t].X for t in range(T)] for i in range(I)]) ; zdR_re = np.array([[zdR[i, t].X for t in range(T)] for i in range(I)])
OBJ_RE = model.objVal

Set parameter MIPGap to value 1e-05
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 2
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-05
PoolSolutions  2
PoolSearchMode  2

Optimize a model with 52900 rows, 34540 columns and 96100 nonzeros
Model fingerprint: 0xde914764
Model has 120 quadratic objective terms
Variable types: 17740 continuous, 16800 integer (16800 binary)
Coefficient statistics:
  Matrix range     [5e-03, 2e+02]
  Objective range  [6e-01, 2e+02]
  QObjective range [2e-07, 2e-07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-03, 8e+02]
Presolve removed 51116 rows and 15434 columns
Presolve time: 0.03s
Presolved: 1784 rows, 19106 columns, 6567 nonzeros
Presolved model has 91 quadratic objective terms
Variab

In [44]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print("\n[REPLAY]") ; print(header)
for t in range(9,20):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")

print("\n[HOLISTIC]") ; print(header)
for t in range(9,20):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[REPLAY]
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 9 |   185.31   113.96     1.82     0.00   131.29   110.10    48.34     0.00    14.31
10 |   445.25   220.98    57.00     0.00   246.14   187.49   108.62     0.00    57.82
11 |   652.06   255.20   214.58     0.00   210.06   183.50   155.71     0.00   155.58
12 |   860.69     0.00   752.36     0.00     0.00     0.00   108.33     0.00   295.72
13 |  1397.66     0.00  1556.27     0.00     0.00     0.00     0.00   158.61   393.22
14 |  1338.06   912.25   115.61     4.10   572.65   522.66   264.31     0.00   216.98
15 |   804.05     0.00  1213.42     0.00     0.00     0.00     0.00   409.38   454.87
16 |   756.56   520.42    78.64     1.13   266.73   108.10     0.00     0.00    -0.00
17 |   738.45     0.00   738.45     0.00     0.00     0.00     0.00     0.00    -0.00
18 |   493.98   261.08   169.58     0.0

In [45]:
# print("\n=== 모든 LDR 계수 (i, t별) ===")
# print("Individual | Time | Variable | 상수항     | R계수    ")
# print("-" * 50)

# for t in range(T):
#     for i in range(I):
#         print(f"{i:10d} | {t:4d} | zc       | {zc0_re[i,t]:8.4f} | {zcR_re[i,t]:7.4f}")
#         print(f"{i:10d} | {t:4d} | zd       | {zd0_re[i,t]:8.4f} | {zdR_re[i,t]:7.4f}")
#         print(f"{i:10d} | {t:4d} | x (1st)  | {x_re[i,t]:8.4f} |    -    ")
#         print("-" * 50)

In [46]:
print("="*50) ; print("AGGREGATOR LOSS ANALYSIS") ; print("="*50)

total_losses = []

for t in range(T):
    scenario_losses = []
    
    for s in range(S):
        total_supply = np.sum(dp_re[:, t, s])
        total_demand = np.sum(dm_re[:, t, s])
        
        if total_supply > total_demand:
            excess_supply = total_supply - total_demand
            # loss = excess * (-lambda_rep[t, s]*S - P_RT[t, s])
            # loss = excess * (P_RT[t, s])
            loss = excess_supply * (P_RT[t, s] - (-lambda_rep[t, s]*S))
        else:
            excess = total_demand - total_supply
            # loss = excess * (P_PN[t, s] - (-lambda_rep[t,s]*S))
            # loss = excess * (P_RT[t, s])
            loss = excess * ((-lambda_rep[t,s]*S) - P_RT[t, s])
        
        scenario_losses.append(loss)
    
    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)
    # print(f"t={t} Average Loss: {avg_loss:.2f}")

overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)

print("[SUMMARY]")
print("Individual Participation Profit", OBJ_IND)
print("Expected Replay Profit", OBJ_RE)
print(f"Total loss across all time periods: {total_loss:.2f}")
print("Realized Profit", OBJ_RE + total_loss)
print("Holistic Profit", OBJ_HOL)

AGGREGATOR LOSS ANALYSIS
[SUMMARY]
Individual Participation Profit 1632097.9322771467
Expected Replay Profit 1715226.3187525035
Total loss across all time periods: -20342.50
Realized Profit 1694883.816907601
Holistic Profit 1689908.8515537635
